# LinkedIn API Test Template

This notebook walks through the full end-to-end flow for the LinkedIn automation API. Replace the placeholder values in the configuration cell before running the steps.

> Prerequisites: Docker services (`web`, `linkedin-service`, `postgres`, `redis`, `celery`) must be running.


In [ ]:
#!/usr/bin/env python3
import os
import time
import json
import sys
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# =========================
# Configuration (replace placeholders)
# =========================
BASE_URL = os.getenv("BASE_URL", "http://localhost:8000")
LI_SERVICE_URL = os.getenv("LI_SERVICE_URL", "http://localhost:5001")

ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "YOUR_ADMIN_EMAIL@example.com")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "YOUR_ADMIN_PASSWORD")

USER_EMAIL = os.getenv("USER_EMAIL", "YOUR_TEST_USER_EMAIL@example.com")
USER_PASSWORD = os.getenv("USER_PASSWORD", "YOUR_TEST_USER_PASSWORD")

LINKEDIN_EMAIL = os.getenv("LINKEDIN_EMAIL", "YOUR_LINKEDIN_LOGIN_EMAIL@example.com")
LINKEDIN_PASSWORD = os.getenv("LINKEDIN_PASSWORD", "YOUR_LINKEDIN_LOGIN_PASSWORD")
LINKEDIN_URL = os.getenv("LINKEDIN_URL", "https://www.linkedin.com/in/YOUR-LINKEDIN-PROFILE/")

# Targets template (fill with your own leads)
TARGETS = [
    {
        "url": "https://www.linkedin.com/in/EXAMPLE-CONTACT/",
        "variables": {"name": "FirstName", "last_name": "LastName"},
    },
]



In [ ]:
# =========================
# Helpers
# =========================
def make_session() -> requests.Session:
    """Session with robust retries for transient network/server hiccups."""
    s = requests.Session()
    retry = Retry(
        total=8,
        connect=8,
        read=8,
        backoff_factor=0.5,
        status_forcelist=[408, 429, 500, 502, 503, 504, 522, 524],
        allowed_methods={"GET", "POST", "PUT", "DELETE", "PATCH", "HEAD", "OPTIONS"},
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    return s


def wait_for(url: str, expect=200, timeout=90, interval=1.0, name="service"):
    """Poll a URL until it responds with the expected status."""
    deadline = time.time() + timeout
    last_err = None
    while time.time() < deadline:
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == expect:
                print(f"[ready] {name} OK at {url}")
                return True
            else:
                last_err = f"HTTP {r.status_code} body={r.text[:200]}"
        except Exception as exc:
            last_err = str(exc)
        time.sleep(interval)
    print(f"[wait_for] {name} not ready: last_err={last_err}")
    return False


def json_or_text(resp: requests.Response):
    try:
        return resp.json()
    except Exception:
        return {"_text": resp.text}


def require_ok(resp: requests.Response, msg="request failed"):
    if not resp.ok:
        body = json_or_text(resp)
        raise RuntimeError(f"{msg}: HTTP {resp.status_code} body={body}")


# Shared session
s = make_session()
s.headers.update({"Content-Type": "application/json"})


In [ ]:
# 1) Wait for services
root_url = urljoin(BASE_URL, "/")
wait_for(root_url, expect=200, timeout=120, name="web-api")

li_health = urljoin(LI_SERVICE_URL, "/health")
wait_for(li_health, expect=200, timeout=120, name="linkedin-service")


In [ ]:
# 2) Admin login
print("[step] admin login")
r = s.post(urljoin(BASE_URL, "/auth/login"), json={"email": ADMIN_EMAIL, "password": ADMIN_PASSWORD}, timeout=15)
require_ok(r, "admin login")
admin_tokens = r.json()
admin_access = admin_tokens["access_token"]
print("[ok] admin access token acquired")


In [ ]:
# 3) Issue registration key
print("[step] create registration key")
r = s.post(
    urljoin(BASE_URL, "/admin/registration-keys"),
    headers={"Authorization": f"Bearer {admin_access}"},
    timeout=15,
)
require_ok(r, "create registration key")
registration_key = r.json()["registration_key"]
print("[ok] registration key created")


In [ ]:
# 4) Register user (proceeds to login on 4xx)
print("[step] register user")
r = s.post(
    urljoin(BASE_URL, "/auth/register"),
    json={"email": USER_EMAIL, "password": USER_PASSWORD, "registration_key": registration_key},
    timeout=20,
)
if 400 <= r.status_code < 500:
    print(f"[info] register user returned {r.status_code}: {json_or_text(r)} -> proceeding to login")
else:
    require_ok(r, "register user")
    print("[ok] user registered")


In [ ]:
# 5) User login
print("[step] user login")
r = s.post(urljoin(BASE_URL, "/auth/login"), json={"email": USER_EMAIL, "password": USER_PASSWORD}, timeout=15)
require_ok(r, "user login")
user_tokens = r.json()
user_access = user_tokens["access_token"]
auth_hdr = {"Authorization": f"Bearer {user_access}", "Content-Type": "application/json"}
print("[ok] user access token acquired")


In [ ]:
# 6) Register outreach profile (idempotent)
print("[step] register outreach profile")
r = s.post(
    urljoin(BASE_URL, "/profiles/outreach/register"),
    headers=auth_hdr,
    json={
        "linkedin_email": LINKEDIN_EMAIL,
        "linkedin_password": LINKEDIN_PASSWORD,
        "linkedin_url": LINKEDIN_URL,
    },
    timeout=30,
)

if r.status_code == 403:
    print("[info] outreach already registered, fetching id")
    r2 = s.get(
        urljoin(BASE_URL, "/profiles/outreach/find-by-email"),
        headers=auth_hdr,
        params={"linkedin_email": LINKEDIN_EMAIL},
        timeout=15,
    )
    require_ok(r2, "find outreach by email")
    outreach_profile_id = r2.json()["id"]
else:
    require_ok(r, "outreach register")
    outreach_profile_id = r.json()["id"]

print(f"[ok] outreach_profile_id={outreach_profile_id}")


In [ ]:
# 7) Create campaign template: connection + message
print("[step] create campaign template (connection + message)")
campaign_template_combo_payload = {
    "name": "Template - Connect Then Message",
    "description": "connection followed by message",
    "steps": [
        {
            "step_number": 1,
            "action": "send_connection",
            "additional_note_template": "Hello {{name}} {{last_name}}, let's connect!",
            "delay_timestamp": "30s",
        },
        {
            "step_number": 2,
            "action": "send_message",
            "message_template": "Great to connect, {{name}} {{last_name}}! Following up.",
            "delay_timestamp": "30s",
        },
    ],
}
r = s.post(
    urljoin(BASE_URL, "/campaigns/templates/create"),
    headers=auth_hdr,
    json=campaign_template_combo_payload,
    timeout=20,
)
require_ok(r, "create campaign template (combo)")
campaign_template_combo_id = r.json()["id"]
print(f"[ok] campaign_template_combo_id={campaign_template_combo_id}")


In [ ]:
# 8) Run campaign: connection + message
print("[step] run campaign (connection + message)")
run_payload_combo = {
    "campaign_template_id": campaign_template_combo_id,
    "outreach_profile_id": outreach_profile_id,
    "target_profiles": TARGETS,
}
r = s.post(
    urljoin(BASE_URL, "/campaigns/run"),
    headers=auth_hdr,
    json=run_payload_combo,
    timeout=30,
)
require_ok(r, "run campaign (connection + message)")
print("[ok] run response (connection + message):", json.dumps(r.json(), indent=2))


## Lead Import Preview

Use the cell below to preview the LinkedIn search results and capture them in the `TARGETS` format before running a full import or campaign.


In [ ]:
# 9) Preview lead import results (no campaign actions)
print("[step] lead import preview")

LEAD_IMPORT_SEARCH_URL = os.getenv(
    "LEAD_IMPORT_SEARCH_URL",
    "https://www.linkedin.com/search/results/people/?keywords=marketing%20director",
)

preview_payload = {
    "outreach_profile_id": outreach_profile_id,
    "search_url": LEAD_IMPORT_SEARCH_URL,
    "max_results": 5,
}
r_preview = s.post(
    urljoin(BASE_URL, "/campaigns/import-search/preview"),
    headers=auth_hdr,
    json=preview_payload,
    timeout=45,
)
require_ok(r_preview, "lead import preview")
preview_response = r_preview.json()
preview_targets = preview_response.get("targets", [])
print("[preview targets] copy/paste friendly list:")
print(json.dumps(preview_targets, indent=2))
print(f"[preview total] {preview_response.get('total', len(preview_targets))} lead(s)")
